# Gloms: OME-Zarr -> precomputed raster + meshes -> Neuroglancer
This is the simplest example -- it downloads a real, published OME-Zarr v0.4 dataset (glomeruli segmentation) and converts it directly to precomputed raster + meshes in one combined function call, since OME-Zarr is already chunked and doesn't need to go through a SpatialData staging step first (unlike OME-TIFF -- see the melanoma/invasive notebooks for that path).


In [ ]:
%load_ext jupyter_black

In [2]:
from pathlib import Path
import subprocess
import hashlib
from dirhash import dirhash

from tissue_map_tools.igneous_converters import (
    from_ome_zarr_04_raster_to_sharded_precomputed_raster_and_meshes,
)
from tissue_map_tools.view import (
    compute_initial_camera_state,
    view_precomputed_in_neuroglancer
)

dataset_path = Path.cwd().parent.parent / "data" / "gloms"
raw_path = dataset_path / "raw"
out_path = dataset_path / "out"
raw_path.mkdir(parents=True, exist_ok=True)
out_path.mkdir(parents=True, exist_ok=True)
precomputed_path = out_path / "gloms_precomputed"

## 1. Download and unzip the published dataset (checksum-guarded, safe to re-run)

In [ ]:
URL = "https://s3.embl.de/spatialdata/raw_data/20_1_gloms.zip"
CHECKSUM_DOWNLOAD = "7857a41d9d4d2914353c9ad0f4ea4ede"
CHECKSUM_UNZIPPED = "927146f7a8cbfcbf9de047a6e1e71226"

download_path = raw_path / Path(URL).name
unzipped_path = raw_path / Path(URL).stem

if (
    not download_path.exists()
    or CHECKSUM_DOWNLOAD != hashlib.md5(download_path.read_bytes()).hexdigest()
):
    subprocess.run(f'curl -o "{download_path}" "{URL}"', shell=True, check=True)

if not unzipped_path.exists() or CHECKSUM_UNZIPPED != dirhash(unzipped_path, "md5"):
    subprocess.run(
        f'unzip -o "{download_path}" -d "{raw_path}"', shell=True, check=True
    )

## 2. Convert directly (OME-Zarr -> precomputed raster + meshes, one call)

In [ ]:
if not (precomputed_path / "info").exists():
    from_ome_zarr_04_raster_to_sharded_precomputed_raster_and_meshes(
        ome_zarr_path=str(unzipped_path / "0"),
        precomputed_path=str(precomputed_path),
    )
    print("Conversion complete.")
else:
    print("Precomputed output already exists -- skipping conversion.")

## 3. Visualize in Neuroglancer

This will open with Neuroglancer's own default framing for the data, you can interact with the view directly.

In [ ]:
# viewer = view_precomputed_in_neuroglancer(data_path=str(precomputed_path))
# viewer

## 4. Visualize in Vitessce
To view the data in Vitessce.
`use_web_app=True` opens vitessce.io in a new browser tab — note this blocks the notebook cell, waiting for you to press Enter in the running kernel's terminal once you're done viewing, before the cell completes.
`use_web_app=False` (or if not provided -- default) opens the Vitessce widget inline in this notebook.

When `segments` and `obsSets.csv` are not provided, all segments from the meshes will be visualized as obsSets.

**Note**: Kernel needs clean restart (by shutting down all kernels) if subsequent running is not displaying the meshes in the notebook.

In [4]:
from tissue_map_tools.view import compute_initial_camera_state
from tissue_map_tools.vitessce_configs.layer_specs import SegmentationLayerSpec
from tissue_map_tools.vitessce_configs.neuroglancer_config_builder import build_neuroglancer_config

use_web_app = None

# Stop previously started local servers before re-running, so repeated runs
# of this cell don't leave orphaned background servers/threads piling up in
# the kernel (which is what was causing meshes not to (re)load without a
# kernel restart).
if 'vc' in dir():
    if use_web_app:
        vc.stop_all_servers()
    else: 
        vc.config.stop_all_servers()
    
initial_camera_state = compute_initial_camera_state(
    data_path=str(precomputed_path),
)

vc = build_neuroglancer_config(
    name="Precomputed data",
    segmentations=[
        SegmentationLayerSpec(file_uid="segmentation", data_path=str(precomputed_path)),
    ],
    initial_camera_state=initial_camera_state,
    use_web_app = use_web_app
)

vc

hi


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:00<00:00, 99.60it/s]
